In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("--- AVVIO CRUSCOTTO QUANTITATIVO: MONITORAGGIO ALPHA LIVE ---")

# 1. CARICAMENTO DATI LIVE
# Sostituisci questo con il percorso reale del tuo file sul Mac/VM
percorso_csv = r"C:\Users\Leona\Documents\NinjaTrader 8\Live_Trades_Log.csv" 

try:
    df_live = pd.read_csv(percorso_csv)
    print(f"File log caricato con successo. Totale trade eseguiti in Forward Test: {len(df_live)}")
except FileNotFoundError:
    raise FileNotFoundError(f"File non trovato in {percorso_csv}. Il bot ha già fatto almeno un trade?")

# 2. CALCOLO METRICHE GLOBALI
tot_trades = len(df_live)
wins = len(df_live[df_live['Esito'] == 'WIN'])
losses = len(df_live[df_live['Esito'] == 'LOSS'])

win_rate_live = (wins / tot_trades) * 100 if tot_trades > 0 else 0
pnl_netto_punti = df_live['PnL_Punti'].sum()
avg_win = df_live[df_live['Esito'] == 'WIN']['PnL_Punti'].mean() if wins > 0 else 0
avg_loss = df_live[df_live['Esito'] == 'LOSS']['PnL_Punti'].mean() if losses > 0 else 0

expectancy_live = (win_rate_live/100 * avg_win) + ((1 - win_rate_live/100) * avg_loss) if tot_trades > 0 else 0

# Baseline Storica (I numeri del tuo Backtest su 4 mesi)
BASELINE_WR = 76.0
BASELINE_EXP = 0.36 # Punti 

# --- REGOLA DELLA SIGNIFICATIVITÀ STATISTICA ---
MIN_TRADES_PER_TEST = 20 # Sotto questa soglia, non giudichiamo l'Alpha

print("\n=== CONFRONTO: BACKTEST vs FORWARD TEST (REALITY CHECK) ===")
print(f"Win Rate:    [Storico: {BASELINE_WR:.2f}%] -> [Live: {win_rate_live:.2f}%]")
print(f"Expectancy:  [Storico: +{BASELINE_EXP:.2f} pt] -> [Live: {expectancy_live:.2f} pt]")
print(f"Avg Win:     {avg_win:.2f} pt")
print(f"Avg Loss:    {avg_loss:.2f} pt (Incluso slippage reale)")

# 3. ANALISI DEGRADO (ALPHA DECAY Z-SCORE)
# Finestra mobile fissa a 10 trade per il grafico
df_live['Rolling_WR'] = df_live['Esito'].apply(lambda x: 1 if x == 'WIN' else 0).rolling(window=min(10, max(1, tot_trades)), min_periods=1).mean() * 100

print("\n=== STATO DEL SISTEMA ===")
if tot_trades < MIN_TRADES_PER_TEST:
    print(f"⚪ STATUS: FASE DI RODAGGIO (Calibrazione).")
    print(f"    -> Hai eseguito solo {tot_trades} trade su {MIN_TRADES_PER_TEST} minimi necessari.")
    print("    -> La varianza statistica è troppo alta. Il sistema sta raccogliendo campioni.")
    print("    -> Ignora temporaneamente le oscillazioni del Win Rate.")
elif win_rate_live >= BASELINE_WR - 10: 
    print("🟢 STATUS: EDGE INTATTO. Il sistema sta performando entro le deviazioni standard attese.")
elif win_rate_live >= 50:
    print("🟡 STATUS: ATTENZIONE (WARNING). Il Win Rate è calato oltre la norma. Possibile mutamento della volatilità.")
else:
    print("🔴 STATUS: ALPHA DECAY RILEVATO. Il sistema sta perdendo l'edge. Richiesta recalibrazione su Python.")

# 4. VISUALIZZAZIONE EQUITY CURVE E WIN RATE ROLLING
if tot_trades > 0:
    plt.figure(figsize=(14, 6))

    # Subplot 1: Equity Curve Punti
    plt.subplot(1, 2, 1)
    df_live['Cumulative_PnL'] = df_live['PnL_Punti'].cumsum()
    plt.plot(df_live.index + 1, df_live['Cumulative_PnL'], marker='o', color='blue', linestyle='-')
    plt.axhline(0, color='red', linestyle='--')
    plt.title('Forward Testing: Equity Curve (Punti)')
    plt.xlabel('Numero Trade')
    plt.ylabel('PnL Cumulativo (Punti)')
    plt.grid(True, alpha=0.3)

    # Subplot 2: Win Rate Mobile
    plt.subplot(1, 2, 2)
    plt.plot(df_live.index + 1, df_live['Rolling_WR'], color='purple', linestyle='-')
    plt.axhline(BASELINE_WR, color='green', linestyle='--', label='Baseline Storica (76%)')
    plt.axhline(50, color='red', linestyle='--', label='Soglia Rischio (50%)')
    
    # Ombreggiatura per la zona di "Rodaggio"
    if tot_trades < MIN_TRADES_PER_TEST:
        plt.axvspan(0, tot_trades + 1, color='grey', alpha=0.2, label='Zona di Rodaggio')
    else:
        plt.axvspan(0, MIN_TRADES_PER_TEST, color='grey', alpha=0.2, label='Zona di Rodaggio')

    plt.title(f'Win Rate Mobile (Ultimi 10 Trade)')
    plt.xlabel('Numero Trade')
    plt.ylabel('Win Rate (%)')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()